# 📊 FinReason — Full Project on Google Colab

**Teaching a Small LLM Financial Numerical Reasoning via GRPO**

EGN 6217 — Applied Deep Learning | Spring 2026

---

### Before you start:
1. Go to **Runtime → Change runtime type → T4 GPU** (free) or **A100** (Pro)
2. Run cells top to bottom in order
3. The full pipeline takes ~4-6 hours on T4, ~2-3 hours on A100

## Step 0: Install Everything

In [ ]:
# Install all dependencies (takes ~3-5 minutes)
!pip install -q torch torchvision torchaudio
!pip install -q transformers>=4.46.0 trl>=0.12.0 peft>=0.13.0
!pip install -q bitsandbytes>=0.44.0 accelerate>=1.0.0
!pip install -q datasets matplotlib seaborn pandas numpy tqdm streamlit
!pip install -q pdfplumber pdf2image pytesseract

# Install Unsloth (saves ~30% VRAM)
!pip install -q unsloth

# Install OCR tools
!apt-get install -y -q tesseract-ocr poppler-utils

print("\n✓ All packages installed")

## Step 1: Upload Project Files

**Option A (recommended):** Upload `FinReason_COMPLETE.zip` from your computer

**Option B:** Clone from GitHub if you've pushed it

In [ ]:
# === OPTION A: Upload the zip ===
from google.colab import files
print("Upload FinReason_COMPLETE.zip when prompted...")
uploaded = files.upload()

!unzip -o FinReason_COMPLETE.zip
%cd FinReason
!ls

In [ ]:
# === OPTION B: Clone from GitHub (uncomment and edit) ===
# !git clone https://github.com/YOUR_USERNAME/FinReason.git
# %cd FinReason

## Step 2: Verify GPU

In [ ]:
import torch
print(f"GPU:    {torch.cuda.get_device_name(0)}")
print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"CUDA:   {torch.version.cuda}")

# Colab T4  = 16 GB  → can use batch=2, G=8
# Colab A100 = 40 GB → can use Qwen2.5-3B, batch=4
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram > 30:
    print("\n🚀 A100 detected! You can use larger models and bigger batches.")
elif vram > 14:
    print("\n✓ T4 detected. Good for the full pipeline.")
else:
    print("\n⚠ Small GPU. May need to reduce batch sizes.")

## Step 3: Test Reward Function

In [ ]:
import sys, os
sys.path.insert(0, 'src')

# Self-test — must show all ✓
!python src/shared_utils.py

## Step 4: Download & Explore FinQA

In [ ]:
!python src/step_01_explore_data.py

## Step 5: Zero-Shot Baseline (Checkpoint 1)

**GO/NO-GO gate.** If accuracy < 2%, switch to a larger model.

Takes ~15-20 minutes on T4.

In [ ]:
!python src/step_02_zeroshot_baseline.py

## Step 6: Format Data for SFT

In [ ]:
!python src/step_03_format_data.py

## Step 7: SFT Training (Stage 1)

~1-2 hours on T4, ~30-45 min on A100.

**Colab tip:** If you have T4/A100, you can increase batch size for faster training.

In [ ]:
# Optional: increase batch size for Colab (more VRAM available)
# Edit src/sft_train.py and change:
#   per_device_train_batch_size=1  →  per_device_train_batch_size=2  (T4)
#   per_device_train_batch_size=1  →  per_device_train_batch_size=4  (A100)

!python src/sft_train.py

## Step 8: Evaluate SFT (Checkpoint 2)

In [ ]:
!python src/step_05_eval_sft.py

## Step 9: GRPO Training (Stage 2) ⭐

**This is the core of the project.**

~2-4 hours on T4, ~1-2 hours on A100.

**Colab tip:** You can increase group size for better signal:
- T4: change `NUM_GEN=4` → `NUM_GEN=8` in `src/grpo_train.py`
- A100: change `NUM_GEN=4` → `NUM_GEN=8` and `MAX_TRAIN=2000` → `MAX_TRAIN=4000`

In [ ]:
!python src/grpo_train.py

## Step 10: Evaluate GRPO (Checkpoint 3 — FINAL)

This prints the comparison of all 3 checkpoints.

In [ ]:
!python src/step_07_eval_grpo.py

## Step 11: Generate Figures

In [ ]:
!python src/analysis.py

## Step 12: View Results

In [ ]:
# Show all figures
from IPython.display import Image, display
import os

for fig in ['results/fig1_accuracy.png', 'results/fig2_grpo_curves.png',
            'results/fig3_error_analysis.png', 'results/fig4_think_analysis.png']:
    if os.path.exists(fig):
        print(f'\n{fig}:')
        display(Image(fig))
    else:
        print(f'  {fig} not found yet')

In [ ]:
# Print final accuracy numbers
import json

print('=' * 50)
print('  FINAL RESULTS')
print('=' * 50)

for name, path in [('Zero-Shot', 'outputs/zeroshot_results.json'),
                    ('SFT',       'outputs/sft_results.json'),
                    ('GRPO',      'outputs/grpo_results.json')]:
    if os.path.exists(path):
        acc = json.load(open(path))['accuracy']
        bar = '█' * int(acc * 60)
        print(f'  {name:>10s}: {acc*100:.2f}% {bar}')

# Show think rate
if os.path.exists('outputs/grpo_results.json'):
    grpo = json.load(open('outputs/grpo_results.json'))
    print(f'\n  Think rate: {grpo.get("think_rate", 0)*100:.1f}%')

## Step 13: Show Qualitative Examples

In [ ]:
# Show examples where GRPO model used <think> reasoning
if os.path.exists('outputs/grpo_results.json'):
    grpo = json.load(open('outputs/grpo_results.json'))
    think_examples = [r for r in grpo['results'] if r.get('think')][:5]
    
    print(f'Examples with <think> reasoning ({len(think_examples)} shown):\n')
    for ex in think_examples:
        print(f"  Q: {ex['q']}")
        print(f"  GT: {ex['gt']}")
        print(f"  Model: {ex['raw'][:300]}...")
        print(f"  Extracted: {ex['pred']}")
        print(f"  {'✓' if ex['ok'] else '✗'}")
        print()

## Step 14: Save Everything to Google Drive

**Important:** Colab deletes files when the session ends. Save to Drive!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create save directory
save_dir = '/content/drive/MyDrive/FinReason_Results'
os.makedirs(save_dir, exist_ok=True)

# Copy results
!cp -r outputs/* "{save_dir}/"
!cp -r results/* "{save_dir}/"

# Copy model checkpoints (these are small — just LoRA adapters ~10-30 MB each)
!cp -r checkpoints/sft/final_adapter "{save_dir}/sft_adapter/" 2>/dev/null
!cp -r checkpoints/grpo/final_adapter "{save_dir}/grpo_adapter/" 2>/dev/null

print(f'\n✓ Everything saved to Google Drive: {save_dir}')
!ls "{save_dir}/"

## Step 15: Download Results to Your Computer

Download the key files to include in your report and GitHub repo.

In [ ]:
# Zip all results for download
!zip -r /content/finreason_results.zip outputs/ results/ checkpoints/sft/final_adapter/ checkpoints/grpo/final_adapter/ 2>/dev/null

from google.colab import files
files.download('/content/finreason_results.zip')
print('\n✓ Download started. Extract into your FinReason folder on your laptop.')

---

## ✓ Done!

You now have:
- `outputs/zeroshot_results.json` — Checkpoint 1
- `outputs/sft_results.json` — Checkpoint 2
- `outputs/grpo_results.json` — Checkpoint 3
- `results/fig1_accuracy.png` through `fig4_think_analysis.png`
- Model adapters in `checkpoints/`

**Next:** Copy results back to your laptop, run `streamlit run ui/app.py` locally for the demo, write the report.